# 04 · Profile: BLS education & training requirements (2025 edition)

**Method:** `docs/framework.md` §3 (the 10-section exploration template). Card: `docs/source-cards/bls_ep_education.md`.

**Why this source matters.** brief Q2 asks *"are graduates ready for the jobs?"*. The first half of that is *"what does the job ask for at the door?"*. BLS Employment Projections assigns every job three entry requirements: typical education, work experience in a related job, and on-the-job training. It also publishes what education the people *already in* each job actually hold. That's the only national, job-by-job answer to "do I need a bachelor's for this?", and the field the crowded-field rule (brief S11) and the associate-vs-bachelor's split depend on.

**What this notebook does:** reads the workbook, checks grain, codes and arithmetic, joins it to the NYC sources from notebooks 02–03, ties it to BLS's own pages, and ends with an issues list and proposed staging tables. It changes nothing.

Every read uses `dtype=str, keep_default_na=False`.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
SOC = r"\d\d-\d{4}"  # a 6-digit SOC-style code

# BLS's own category order, lowest to highest (EP technical documentation)
EDU_ORDER = ["No formal educational credential", "High school diploma or equivalent", "Some college, no degree",
             "Postsecondary nondegree award", "Associate's degree", "Bachelor's degree", "Master's degree",
             "Doctoral or professional degree"]
EXP_ORDER = ["None", "Less than 5 years", "5 years or more"]
OJT_ORDER = ["None", "Short-term on-the-job training", "Moderate-term on-the-job training",
             "Long-term on-the-job training", "Apprenticeship", "Internship/residency"]

## 0. Source header

| | BLS Employment Projections: education and training (`bls_ep_education`) |
|---|---|
| **Publisher** | U.S. Bureau of Labor Statistics, Employment Projections (EP) program |
| **Dataset** | `education.xlsx`, the education tables of the 2025–35 projections (2025 National Employment Matrix). Linked from [EP tables](https://www.bls.gov/emp/tables.htm) |
| **What's inside** | Table 5.1 earnings and unemployment by attainment (CPS 2025) · 5.2 national jobs by entry education, 2025 and 2035 · 5.3 attainment of current workers by job (ACS 2023–24) · 5.4 entry education, experience and training by job |
| **How it was collected** | 5.4 is an expert assignment, not a survey: BLS economists pick the *typical* path into each job, using ACS attainment, O\*NET requirements, NCES completions and input from employers, educators and unions. Legal requirements (physicians, lawyers) override. Where several paths exist, only the typical one is recorded ([EP education technical documentation](https://www.bls.gov/emp/documentation/education/tech.htm)) |
| **Geography** | **United States.** There is no NYC or state version |
| **Editions** | Re-assigned every year with the projections. This file = 2025 assignments |
| **License** | U.S. government work, public domain |

**Note:** the prototype's `data/raw/education.xlsx` is checked below: it's the same file.

In [2]:
man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
FILE = man[man.key == "bls_ep_education"][["key", "file", "downloaded_utc", "sha256", "note"]]
display(FILE)
PATH = RAW / FILE.file.iloc[0]

import hashlib
proto = RAW / "education.xlsx"
print("Prototype copy identical to the manifest file:",
      hashlib.sha256(proto.read_bytes()).hexdigest() == FILE.sha256.iloc[0])

,key,file,downloaded_utc,sha256,note
16,bls_ep_education,jobs/bls_ep_education/education_2026-10-12.xlsx,2026-10-12T17:05:26+00:00,67b78dbccc8d6002e6c30f39a75d381380e2e9347bd2d87d71c3c98e7f36a531,"Entry education, experience, training; attainment table 5.3"


Prototype copy identical to the manifest file: True


## 1. Load as text

The workbook has an index sheet and four tables. Each table has a title in row 1, headers in row 2, and notes and footnotes below the data. Read each sheet raw first to see that layout.

In [3]:
xl = pd.ExcelFile(PATH)
raw = {s: pd.read_excel(PATH, sheet_name=s, dtype=str, header=None, keep_default_na=False) for s in xl.sheet_names}
for s, d in raw.items():
    print(f"{s:<10} {d.shape[0]:>4} rows x {d.shape[1]} cols | title: {d.iloc[0, 0][:110]}")

Index         5 rows x 1 cols | title: Education.xlsx contents
Table 5.1    15 rows x 3 cols | title: Table 5.1 Unemployment rates and earnings by educational attainment, 2025
Table 5.2    15 rows x 8 cols | title: Table 5.2 Employment, wages, and projected change in employment by typical entry-level education, 2025 and pro
Table 5.3   837 rows x 9 cols | title: Table 5.3 Educational attainment for workers 25 years and older by detailed occupation, 2023–24 (Percent)
Table 5.4   834 rows x 6 cols | title: Table 5.4 Education and training assignments by detailed occupation, 2025


In [4]:
def table(sheet):
    # Header in row 2; keep only rows whose code column looks like a SOC code
    d = pd.read_excel(PATH, sheet_name=sheet, dtype=str, header=1, keep_default_na=False)
    d.columns = [c.strip() for c in d.columns]
    is_row = d.iloc[:, 1].str.fullmatch(SOC)
    print(f"{sheet}: {is_row.sum()} data rows; dropped trailing text rows: {d.loc[~is_row].iloc[:, 0].str[:100].tolist()}")
    return d[is_row].reset_index(drop=True)

t54 = table("Table 5.4")
t53 = table("Table 5.3")
t54.columns = ["title", "soc", "education", "experience", "training", "ooh"]
t53 = t53.rename(columns={t53.columns[0]: "title", t53.columns[1]: "soc"})
ATT = list(t53.columns[2:])
print("\nTable 5.3 attainment columns:", ATT)

padded = {c: (t54[c] != t54[c].str.strip()).sum() for c in t54.columns}
print("Table 5.4 cells with stray spaces:", {k: v for k, v in padded.items() if v} or "none")

Table 5.4: 831 data rows; dropped trailing text rows: ['Source: U.S. Bureau of Labor Statistics.']
Table 5.3: 832 data rows; dropped trailing text rows: ['[1] This occupation is not an exact match with ACS; see https://www.bls.gov/emp/classifications-cros', 'Note: Data are from the 2023 and 2024 American Community Survey Public Use Microdata, U.S. Departmen', 'Source: U.S. Bureau of Labor Statistics.']

Table 5.3 attainment columns: ['Less than high school diploma', 'High school diploma or equivalent', 'Some college, no degree', "Associate's degree", "Bachelor's degree", "Master's degree", 'Doctoral or professional degree']
Table 5.4 cells with stray spaces: none


**Load findings**
- Data rows are easy to isolate: every real row has a SOC-style code in column B. The trailing rows are notes, the `Source:` line and, in 5.3, the footnote `[1]` text.
- 5.4 has 831 jobs; 5.3 has 832 rows (the same 831 + the `00-0000` total).
- Tables 5.3 and 5.4 are clean: no padding, no blanks in the data rows. Table 5.2 indents its category labels with two spaces (`'  Bachelor\'s degree'`), so they must be stripped before joining to 5.4's labels. The OOH column uses an em dash `—` for "no Handbook page".

## 2. What one row is (grain)

**Presumed key:** `soc` (one row per job, one edition).

In [5]:
print("Table 5.4 duplicate SOC:", t54.soc.duplicated().sum(), "| rows:", len(t54))
print("Table 5.3 duplicate SOC:", t53.soc.duplicated().sum(), "| rows:", len(t53))
print("In 5.3 but not 5.4:", sorted(set(t53.soc) - set(t54.soc)), "| in 5.4 but not 5.3:", sorted(set(t54.soc) - set(t53.soc)))
same_title = (t53.set_index("soc").title.str.replace(r"\[\d\]", "", regex=True).str.strip()
              .reindex(t54.soc).values == t54.title.values).mean()
print(f"Titles agree between 5.3 and 5.4 (after removing footnote marks): {same_title:.1%}")

Table 5.4 duplicate SOC: 0 | rows: 831
Table 5.3 duplicate SOC: 0 | rows: 832
In 5.3 but not 5.4: ['00-0000'] | in 5.4 but not 5.3: []
Titles agree between 5.3 and 5.4 (after removing footnote marks): 100.0%


**Grain findings:** `soc` is unique in both tables . The two tables cover exactly the same 831 jobs, and their titles agree once the `[1]` footnote marks are removed from 5.3. One row = one job, nationally, for the 2025 edition. There is no area or year column: those come from the file's edition.

## 3. Column profile

Table 5.4 is three categorical columns. Count every value, in BLS's order:

In [6]:
for col, order in [("education", EDU_ORDER), ("experience", EXP_ORDER), ("training", OJT_ORDER)]:
    vc = t54[col].value_counts()
    unknown = sorted(set(vc.index) - set(order))
    display(vc.reindex(order).rename(f"jobs ({col})").to_frame().T)
    assert not unknown, f"values outside BLS's list: {unknown}"
print("OOH column:", t54.ooh.value_counts().to_dict())

education,No formal educational credential,High school diploma or equivalent,"Some college, no degree",Postsecondary nondegree award,Associate's degree,Bachelor's degree,Master's degree,Doctoral or professional degree
jobs (education),109,326,7,51,48,177,40,73


experience,None,Less than 5 years,5 years or more
jobs (experience),716,87,28


training,None,Short-term on-the-job training,Moderate-term on-the-job training,Long-term on-the-job training,Apprenticeship,Internship/residency
jobs (training),321,174,231,54,15,36


OOH column: {'OOH Content': 606, '—': 225}


In [7]:
# Table 5.3: percentages that should sum to 100 per job
att = t53.set_index("soc")[ATT].apply(pd.to_numeric, errors="coerce")
print("Non-numeric attainment cells:", int(att.isna().sum().sum()))
print("Row sums:", att.sum(axis=1).round(1).value_counts().sort_index().to_dict())

Non-numeric attainment cells: 0
Row sums: {99.8: 14, 99.9: 202, 100.0: 456, 100.1: 142, 100.2: 18}


**Profile findings**
- Every job has all three assignments, and every value is on BLS's published list. They're ordered categories, so staging stores a rank next to the label (e.g. Bachelor's = 6 of 8).
- **Most jobs need no experience** (716 of 831). Only 115 ask for prior work in a related job, and 28 of those for 5+ years. These are mostly managers and supervisors: jobs you are *promoted into*, not hired into from school.
- **Postsecondary nondegree award** (certificates, 51 jobs) sits *below* the associate's degree in BLS's order, but CUNY awards both. It matters for the associate/certificate side of the brief.
- Table 5.3 is complete: no blanks, and every row sums to 100 ± 0.2 (rounding).

## 4. Codes, hierarchy and special values

**(a) Which code system?** The header says *2025 National Employment Matrix code*. That's SOC 2018 with a few BLS-only aggregates. Check for the hybrid codes seen in notebooks 02–03, and for codes ending in `0`, which in SOC are broad groups, not jobs.

In [8]:
codes = set(t54.soc)
probe = {"15-1256 (hybrid)": "15-1256", "13-2098 (hybrid)": "13-2098", "15-1252 (SOC 2018)": "15-1252",
         "15-1253 (SOC 2018)": "15-1253", "13-2051 (SOC 2018)": "13-2051", "15-1132 (SOC 2010)": "15-1132"}
print({k: v in codes for k, v in probe.items()})
print("Codes ending in 8:", sorted(c for c in codes if c.endswith("8")))
t54[t54.soc.str.endswith("0")][["soc", "title", "education"]]

{'15-1256 (hybrid)': False, '13-2098 (hybrid)': False, '15-1252 (SOC 2018)': True, '15-1253 (SOC 2018)': True, '13-2051 (SOC 2018)': True, '15-1132 (SOC 2010)': False}
Codes ending in 8: ['17-3028', '21-1018', '25-2058', '29-1128', '29-1218', '49-2098', '49-9098', '51-2028', '51-9198']


,soc,title,education
38,13-1020,Buyers and purchasing agents,Bachelor's degree
56,13-2020,Property appraisers and assessors,Bachelor's degree
349,29-2010,Clinical laboratory technologists and technicians,Bachelor's degree
375,31-1120,Home health and personal care aides,High school diploma or equivalent
469,39-7010,Tour and travel guides,High school diploma or equivalent
610,47-4090,Miscellaneous construction and related workers,High school diploma or equivalent
684,51-2090,Miscellaneous assemblers and fabricators,High school diploma or equivalent


(b) Footnote `[1]` in Table 5.3: borrowed attainment. BLS footnotes 458 titles as *"not an exact match with ACS"*. The ACS (Census) codes jobs less finely than SOC, so several SOC jobs share one ACS code and therefore one attainment profile. How many rows are copies?

In [9]:
t53["acs_inexact"] = t53.title.str.contains(r"\[1\]")
prof = t53[t53.soc != "00-0000"].copy()
prof["profile"] = prof[ATT].agg("|".join, axis=1)
prof["shared_by"] = prof.groupby("profile").soc.transform("size")
print("Rows footnoted [1]:", prof.acs_inexact.sum(), "of", len(prof))
print("Rows whose 7 percentages exactly equal another job's:", (prof.shared_by > 1).sum(),
      "| distinct profiles:", prof.profile.nunique())
print("Footnoted but unique profile:", (prof.acs_inexact & (prof.shared_by == 1)).sum(),
      "| shared but not footnoted:", (~prof.acs_inexact & (prof.shared_by > 1)).sum())
display(prof.sort_values("shared_by", ascending=False)[["soc", "title", "shared_by"] + ATT].head(6))

Rows footnoted [1]: 458 of 831
Rows whose 7 percentages exactly equal another job's: 454 | distinct profiles: 520
Footnoted but unique profile: 4 | shared but not footnoted: 0


,soc,title,shared_by,Less than high school diploma,High school diploma or equivalent,"Some college, no degree",Associate's degree,Bachelor's degree,Master's degree,Doctoral or professional degree
213,25-1062,"Area, ethnic, and cultural studies teachers, postsecondary[1]",37,0.7,1.8,2,1.7,14.1,29.8,49.9
216,25-1065,"Political science teachers, postsecondary[1]",37,0.7,1.8,2,1.7,14.1,29.8,49.9
215,25-1064,"Geography teachers, postsecondary[1]",37,0.7,1.8,2,1.7,14.1,29.8,49.9
217,25-1066,"Psychology teachers, postsecondary[1]",37,0.7,1.8,2,1.7,14.1,29.8,49.9
218,25-1067,"Sociology teachers, postsecondary[1]",37,0.7,1.8,2,1.7,14.1,29.8,49.9
204,25-1032,"Engineering teachers, postsecondary[1]",37,0.7,1.8,2,1.7,14.1,29.8,49.9


**Code findings**
- No hybrid or SOC 2010 codes. The 2025 matrix is SOC 2018. The hybrid aggregates in the 2022-32 projections and OEWS 2019–20 (`15-1256`, `13-2098` …) are not here, so those rows need the notebook 06 bridge before they can get an education value.
- 7 BLS aggregate codes end in `0` (`13-1020` Buyers and purchasing agents, `29-2010` Clinical lab technologists *and* technicians, `31-1120` Home health *and* personal care aides …). In SOC 2018 these are broad groups; BLS projects them as one job because OEWS can't split them. NYS DOL wages use the same codes (§6), so they join as they are.
- A code ending in `8` is not a hybrid marker. Of the 9 codes ending in `8` here, 7 are genuine SOC 2018 detailed jobs (`29-1218` OB-GYNs, `51-9198` production helpers …): SOC 2018 itself used `8` for jobs it merged. *(Corrected in notebook 06: the other two, `21-1018` mental health counselors and `51-2028` assemblers, are BLS aggregates, not SOC 2018 codes.)* Notebook 03 §4 counted "codes ending in 8" as a hybrid signal. The year → edition result there still holds (it rests on the marker codes), but notebook 06 must detect hybrids from BLS's hybrid list, not from the last digit.
- More than half the attainment rows are borrowed. 458 of 831 are footnoted, and 454 of them share their exact profile with at least one other job: 831 jobs have only 520 distinct profiles. The extreme: 37 postsecondary teacher jobs (engineering, psychology, geography …) all carry one identical profile. Treatment: keep `acs_inexact` as a flag, and never present a footnoted job's attainment as if it were measured for that job alone.

## 5. Coverage

National only, one edition. Check what each table covers and the national mix of jobs by entry education (Table 5.2):

In [10]:
t52 = pd.read_excel(PATH, sheet_name="Table 5.2", dtype=str, header=1, keep_default_na=False)
t52 = t52[t52.iloc[:, 1].str.fullmatch(r"[\d.]+")].reset_index(drop=True)
t52.columns = [c.strip() for c in t52.columns]
t52.iloc[:, 0] = t52.iloc[:, 0].str.strip()
display(t52)
print("Table 5.2 notes:", *raw["Table 5.2"].iloc[11:, 0], sep="\n  ")
print("\nTable 5.1 notes:", *raw["Table 5.1"].iloc[11:, 0], sep="\n  ")

,Typical education needed for entry,"Employment, 2025","Employment, 2035","Employment distribution, percent, 2025","Employment distribution, percent, 2035","Employment change, numeric, 2025–35","Employment change, percent, 2025–35","Median annual wage, dollars, 2025"
0,"Total, all occupations",170280.8,176198.5,100,100,5917.6,3.5,50980
1,Doctoral or professional degree,4543.2,4833.6,2.7,2.7,290.5,6.4,128870
2,Master's degree,3941.3,4376.7,2.3,2.5,435.4,11,83210
3,Bachelor's degree,42870.1,45292,25.2,25.7,2421.9,5.6,96480
4,Associate's degree,3554.1,3728.7,2.1,2.1,174.6,4.9,64990
5,Postsecondary nondegree award,10679.1,11284.6,6.3,6.4,605.5,5.7,49910
6,"Some college, no degree",4172.9,4039.7,2.5,2.3,-133.2,-3.2,46670
7,High school diploma or equivalent,60319,61503.4,35.4,34.9,1184.4,2,48050
8,No formal educational credential,40201.2,41139.8,23.6,23.3,938.6,2.3,36400


Table 5.2 notes:
  Note:
  Wage data are from the Occupational Employment and Wage Statistics program. The wage data cover non-farm wage and salary workers and do not cover the self-employed, owners and partners in unincorporated firms, or household workers.
  The data shown in this table include all jobs from 2025–35, not just entry-level jobs. Entry-level education reflects 2025 requirements—BLS does not project educational requirements.
  Source: U.S. Bureau of Labor Statistics.

Table 5.1 notes:
  Note:
  Data are from the Current Population Survey. Data are for persons age 25 and over. Earnings are for full-time wage and salary workers.
  Annual estimates for 2025 are 11-month averages that exclude October. (Data for October 2025 were not collected due to the federal government shutdown.) As a result, 2025 annual estimates are not strictly comparable with annual averages for other years.
  Source: U.S. Bureau of Labor Statistics.


**Coverage findings**
- **United States only.** BLS assigns requirements nationally. NYC employers may ask for more (a crowded market lets them, see §8), but no public source measures that by job for NYC. *This is a national rule applied to NYC jobs.*
- One assignment per job, no history. Earlier editions would show whether a job's requirement changed (e.g. a job moving from associate's to bachelor's). Not needed for v1; note for later.
- Three time references in one file: assignments = 2025; attainment (5.3) = ACS 2023–24; earnings (5.1) = CPS 2025, an 11-month average because October 2025 was not collected (federal shutdown), so BLS warns 2025 isn't strictly comparable with other years.
- Nationally, a quarter of jobs (25.2%) need a bachelor's to enter and 2.1% an associate's. Table 5.2 counts *all* jobs in each category, not just openings for new entrants.

## 6. Relationships: will it join?

Two joins matter now: (1) NYS DOL NYC wages and employment (notebook 03), which gives every NYC job an employment weight; (2) the NYC 2022-32 projections (notebook 02). A parent code is dropped when its children are present, so only leaves are compared.

In [11]:
def leaves(codes):
    # Drop a code when a more detailed code in the same set rolls up to it
    s = set(codes)
    def is_parent(c):
        for zeros in ("0000", "000", "00", "0"):
            if c.endswith(zeros):
                stem = c[: 7 - len(zeros)]
                return any(o != c and o.startswith(stem) for o in s)
        return False
    return [c for c in codes if not is_parent(c) and c != "00-0000"]

w = pd.read_csv(RAW / man[man.key == "nysdol_oews_regions"].file.iloc[0], dtype=str, keep_default_na=False)
w.columns = [c.strip() for c in w.columns]
w = w.apply(lambda s: s.str.strip())
nyc = w[w["Area Name"] == "New York City"].rename(columns={"Standard Occupational Code": "soc"})
nyc = nyc[nyc.soc.isin(leaves(nyc.soc))].assign(employment=lambda d: pd.to_numeric(d.Employment, errors="coerce"))
hit = nyc.soc.isin(codes)
print(f"NYS DOL NYC wage jobs (leaves): {len(nyc)} | with a BLS assignment: {hit.sum()} ({hit.mean():.1%}) "
      f"| share of NYC employment covered: {nyc.employment[hit].sum() / nyc.employment.sum():.1%}")

pr = pd.read_csv(RAW / man[man.key == "nysdol_projections"].file.iloc[0], dtype=str, keep_default_na=False)
pr = pr[pr.Area == "New York City"].assign(soc=lambda d: d.SOC.str.lstrip("'"))
pr = pr[pr.soc.isin(leaves(pr.soc))]
ph = pr.soc.isin(codes)
print(f"NYC 2022-32 projection leaves: {len(pr)} | with a BLS assignment: {ph.sum()} ({ph.mean():.1%})")
pr.loc[~ph, ["soc", "Occupation Title", "Base Year"]]

NYS DOL NYC wage jobs (leaves): 679 | with a BLS assignment: 679 (100.0%) | share of NYC employment covered: 100.0%
NYC 2022-32 projection leaves: 543 | with a BLS assignment: 528 (97.2%)


,soc,Occupation Title,Base Year
4441,11-9198,"Personal Service Managers; Entertainment & Recreation Managers, Ex...",16900
4470,13-2098,"Financial and Investment Analysts, Financial Risk Specialists, and...",590
4481,15-1256,Software Developers and Software Quality Assurance Analysts and Te...,600
4482,15-1257,Web Developers and Digital Interface Designers,730
4526,19-3031,"Clinical, Counseling, and School Psychologists",1390
4618,25-3097,"Teachers and Instructors, All Other, Except Substitute Teachers",1690
4651,27-2090,"Miscellaneous Entertainers and Performers, Sports and Related Workers",690
4698,29-1228,"Physicians, All Other; and Ophthalmologists, Except Pediatric",900
4700,29-1298,Acupuncturists and Healthcare Diagnosing or Treating Practitioners...,490
4791,39-1098,First-Line Supervisors of Personal Service & Entertainment and Rec...,1860


In [12]:
# NYC jobs by entry requirement, weighted by NYC employment, next to the national mix (Table 5.2)
m = nyc[hit].merge(t54, on="soc")
nat = t52.set_index(t52.columns[0])["Employment distribution, percent, 2025"].astype(float) / 100
mix = pd.DataFrame({
    "NYC jobs (NYS DOL employment)": m.groupby("education").employment.sum() / m.employment.sum(),
    "US jobs (Table 5.2)": nat,
}).reindex(EDU_ORDER)
display((mix * 100).round(1))
display((m.groupby("experience").employment.sum() / m.employment.sum() * 100).reindex(EXP_ORDER).round(1).rename("NYC share, %").to_frame())

,NYC jobs (NYS DOL employment),US jobs (Table 5.2)
No formal educational credential,17.4,23.6
High school diploma or equivalent,37.3,35.4
"Some college, no degree",2.5,2.5
Postsecondary nondegree award,4.1,6.3
Associate's degree,1.6,2.1
Bachelor's degree,30.7,25.2
Master's degree,2.5,2.3
Doctoral or professional degree,3.8,2.7


,"NYC share, %"
experience,
None,81.4
Less than 5 years,11.6
5 years or more,7.0


**Join findings**
- NYS DOL NYC wages → BLS: every NYC job matches (all 679 leaves, 100% of NYC employment). Both use the same 2025-era SOC 2018 + BLS aggregate list, so this join is clean with no bridge.
- NYC projections 2022-32 → BLS: 528 of 543 leaves match. The 15 misses are the same families as before: the hybrid aggregates (`15-1256`, `15-1257`, `13-2098`, `11-9198` …), three minor groups published as leaves because their detail is suppressed (`45-1000`, `45-4000`, `47-5000`), and three SOC 2018 codes BLS has since split (`19-3031` psychologists, `25-3097`, `53-3058`). Notebook 06's bridge has to cover these, or they lose their education value. Software Developers is among them, so this matters.
- NYC is a more degree-heavy job market than the US: ~31% of NYC jobs need a bachelor's at entry vs 25% nationally. Associate's-level jobs are a thin slice in both (~1.6% NYC, 2.1% US). For CUNY's associate graduates that's a headline: few jobs list the associate's as *the typical* entry, so their path runs through jobs whose typical entry is lower or higher (see §8).
- About 81% of NYC jobs need no prior experience, by BLS's rule.

## 7. Tie-out

(a) Against BLS's Occupational Outlook Handbook. The OOH page for [Software Developers](https://www.bls.gov/ooh/computer-and-information-technology/software-developers.htm) (2025–35 edition, checked) shows: *Typical entry-level education* Bachelor's degree, *Work experience in a related occupation* None, *On-the-job training* None. The EP technical documentation names Registered Nurses = Bachelor's degree as its example of a job with several paths (associate's or diploma also possible).

In [13]:
published = {"15-1252": ("Bachelor's degree", "None", "None"),
             "29-1141": ("Bachelor's degree", None, None)}
for soc, (edu, exp, ojt) in published.items():
    r = t54.set_index("soc").loc[soc]
    ok = r.education == edu and (exp is None or r.experience == exp) and (ojt is None or r.training == ojt)
    print(f"{soc} {r.title:<45} file: {r.education} / {r.experience} / {r.training}  -> {'match' if ok else 'MISMATCH'}")
    assert ok

15-1252 Software developers                           file: Bachelor's degree / None / None  -> match
29-1141 Registered nurses                             file: Bachelor's degree / None / None  -> match


**(b) Internal arithmetic.** Table 5.2's categories should add up to its total, and its percentages to 100. (Percent change and median wage aren't additive, so they're left out.)

In [14]:
v = t52.set_index(t52.columns[0]).astype(float)
v = v[[c for c in v.columns if c.startswith("Employment") and "percent, 2025–35" not in c]]  # additive columns only
parts, total = v.drop(index="Total, all occupations"), v.loc["Total, all occupations"]
chk = pd.DataFrame({"sum of categories": parts.sum(), "published total": total})
chk["difference"] = chk.iloc[:, 0] - chk.iloc[:, 1]
chk.round(1)

,sum of categories,published total,difference
"Employment, 2025",170280.9,170280.8,0.1
"Employment, 2035",176198.5,176198.5,0.0
"Employment distribution, percent, 2025",100.1,100.0,0.1
"Employment distribution, percent, 2035",99.9,100.0,-0.1
"Employment change, numeric, 2025–35",5917.7,5917.6,0.1


(c) Attainment vs requirement: the sanity check that also answers a question. If the assignments are sensible, most workers in a job should hold at least its entry education. Measure, per job, the share of workers *below* and *above* the requirement (only for jobs whose requirement maps onto an ACS attainment level).

In [15]:
# Map entry education onto the ACS attainment columns of Table 5.3 (no ACS level for certificates)
LEVEL = {"No formal educational credential": 0, "High school diploma or equivalent": 1, "Some college, no degree": 2,
         "Associate's degree": 3, "Bachelor's degree": 4, "Master's degree": 5, "Doctoral or professional degree": 6}
a = t53.set_index("soc")[ATT].astype(float)
rows = []
for _, r in t54.iterrows():
    if r.education not in LEVEL:
        continue
    k = LEVEL[r.education]
    rows.append({"soc": r.soc, "title": r.title, "education": r.education,
                 "below_req_pct": a.loc[r.soc].iloc[:k].sum(), "above_req_pct": a.loc[r.soc].iloc[k + 1:].sum()})
fit = pd.DataFrame(rows).merge(nyc[["soc", "employment"]], on="soc", how="left")
display(fit.groupby("education")[["below_req_pct", "above_req_pct"]].median().reindex([e for e in EDU_ORDER if e in LEVEL]).round(1))
rn = a.loc["29-1141"]
rn_nyc = int(nyc.set_index("soc").employment["29-1141"])
print(f"Registered nurses (entry: bachelor's): associate's {rn.iloc[3]:.1f}%, below a bachelor's {rn.iloc[:4].sum():.1f}%, "
      f"bachelor's or more {rn.iloc[4:].sum():.1f}% | NYC employment {rn_nyc:,}")
print("Biggest NYC bachelor's-entry jobs where many workers have less than a bachelor's:")
display(fit[(fit.education == "Bachelor's degree")].nlargest(60, "employment").nlargest(8, "below_req_pct")[["soc", "title", "employment", "below_req_pct"]])
print("Biggest NYC high-school-entry jobs where many workers have a bachelor's or more (crowding signal):")
hs = fit[fit.education == "High school diploma or equivalent"].copy()
hs["ba_plus_pct"] = hs.soc.map(a[ATT[4:]].sum(axis=1))
display(hs.nlargest(60, "employment").nlargest(8, "ba_plus_pct")[["soc", "title", "employment", "ba_plus_pct"]])

,below_req_pct,above_req_pct
education,,
No formal educational credential,0.0,80.7
High school diploma or equivalent,8.4,53.9
"Some college, no degree",25.8,48.5
Associate's degree,38.8,36.3
Bachelor's degree,24.4,26.8
Master's degree,42.6,12.9
Doctoral or professional degree,50.1,0.0


Registered nurses (entry: bachelor's): associate's 24.9%, below a bachelor's 30.5%, bachelor's or more 69.5% | NYC employment 89,560
Biggest NYC bachelor's-entry jobs where many workers have less than a bachelor's:


,soc,title,employment,below_req_pct
258,25-9044,"Teaching assistants, postsecondary",9460.0,65.5
18,11-9021,Construction managers,4900.0,65.3
38,13-1020,Buyers and purchasing agents,15590.0,57.0
1,11-1021,General and operations managers,99410.0,53.1
343,29-2010,Clinical laboratory technologists and technicians,7460.0,49.2
76,15-1244,Network and computer systems administrators,8940.0,47.2
51,13-1151,Training and development specialists,12530.0,46.5
82,15-1299,"Computer occupations, all other",6720.0,43.4


Biggest NYC high-school-entry jobs where many workers have a bachelor's or more (crowding signal):


,soc,title,employment,ba_plus_pct
451,41-3011,Advertising sales agents,10830.0,65.7
248,25-3021,Self-enrichment teachers,17370.0,61.4
440,39-9031,Exercise trainers and group fitness instructors,9360.0,58.7
66,13-2082,Tax preparers,4470.0,56.4
455,41-3091,"Sales representatives of services, except advertising, insurance, ...",46890.0,56.0
461,41-9022,Real estate sales agents,5450.0,53.2
457,41-4012,"Sales representatives, wholesale and manufacturing, except technic...",23920.0,49.6
39,13-1031,"Claims adjusters, examiners, and investigators",5550.0,49.2


**Tie-out findings**
- **(a) ** Software Developers and Registered Nurses match BLS's published pages.
- **(b) ** Table 5.2 categories add to the total (within rounding of the thousands shown).
- **(c)** The assignments behave as expected: in the median bachelor's-entry job, about 3 in 4 workers hold a bachelor's or more (24% below), and in the median high-school-entry job only 8% are below. But the spread is wide, and both tails are useful to the brief:
  - Jobs whose requirement is higher than many current workers have (e.g. Registered Nurses: 24.9% of today's RNs hold an associate's as their highest degree, and NYC has 89,560 RNs). That's the *requirement drifting up*, and it's exactly the CUNY associate-RN situation. The dashboard must show "typical entry: bachelor's" *together with* "X% of current workers have an associate's", or it misleads associate graduates.
  - Jobs whose requirement is lower than what workers hold (e.g. services sales representatives: high-school entry, 56% of workers have a BA+, 46,890 NYC jobs). That's underemployment, a direct input to the crowded-field rule (brief S11): a graduate competing for those jobs isn't using the degree.

## 8. Fitness for purpose

| Question | Fit | Why |
|---|---|---|
| Q2: what education does a job need at entry | Use | The standard answer, one per job, every NYC job covered. Caveats: national; the *typical* path only (RN shows the cost of that); an expert judgment, not a measurement |
| Q2: does it need experience (junior vs senior) | Use, as one input | `experience` separates jobs you enter from school (`None`) from jobs you're promoted into (managers, supervisors). It works per job, not per opening. It doesn't tell junior from senior *openings* within a job, which is the open junior/senior research question |
| **Q2: training after hire** | yes | Apprenticeship and internship/residency flag jobs with a structured on-ramp (electricians, physicians, teachers) |
| Q3 link: which degree level feeds which job | yes | The level filter between majors and jobs: an associate graduate and a bachelor graduate of the same major point to different jobs. Replaces the prototype's hard-coded `EDUCATION_GROUP` with BLS's full 8 levels |
| **Crowded-field rule (S11): underemployment** | Use with limits | Attainment vs requirement shows over-qualified workers per job. But over half the attainment rows are borrowed from a broader ACS group, the ACS is 2023–24 and national, and NYC may differ (ACS PUMS, notebook 08, can measure NYC directly) |
| "Education pays" context (Table 5.1) | partial | National CPS 2025, shutdown-affected. Fine as a one-line context number, not a dashboard measure |
| Anything NYC-specific about requirements | no | No such public source. Say so on the page |

## 9. Issues list and proposed staging tables

| # | Issue | Rows affected | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | Notes, source line and footnote text below the data | 1 row (5.4), 3 rows (5.3) | Keep rows whose code matches `NN-NNNN`; assert 831 / 832 |
| 2 | Footnote mark `[1]` inside 5.3 titles | 458 titles | Strip from the title; keep as `acs_inexact = true` |
| 3 | Borrowed attainment profiles (several jobs share one ACS profile) | 454 jobs (520 distinct profiles for 831 jobs) | Keep the flag; dashboard says "based on a broader group" for those jobs |
| 4 | Categories are text with a natural order | all 831 × 3 | Add `education_rank` (1–8), `experience_rank` (1–3), `training_rank` (1–6) from BLS's order; assert no unknown labels |
| 5 | OOH column uses `—` for none | 225 rows | `has_ooh_page` boolean |
| 6 | 7 BLS aggregate codes ending in `0` | 7 jobs | Keep as is; they match NYS DOL wages exactly. Mark `is_bls_aggregate` in the job dimension |
| 7 | Hybrid / split codes in the 2022-32 projections have no assignment | 15 NYC projection leaves (incl. Software Developers 15-1256) | Resolve through the notebook 06 bridge; for an aggregate, take its children's requirement (they agree for 15-1256 → 15-1252/15-1253) and flag disagreements |
| 8 | National, not NYC | all | Area = United States in the fact table; page footnote |
| 9 | Three time references (assignments 2025, ACS 2023–24, CPS 2025 with October missing) | all | Store `edition = 2025` and `attainment_years = '2023-24'` columns; don't stage 5.1 as a fact |
| 10 | "Typical" path hides alternatives (RN: associate's also accepted) | judgment calls, e.g. 29-1141 | Show current-worker attainment next to the requirement; consider a small hand-kept `alternative_entry` note for CUNY-relevant jobs (decision needed, not now) |
| 11 | Prototype's `data/raw/education.xlsx` duplicates the manifest file | 1 file | Same bytes (§0). Read only the manifest path; leave the prototype file alone |

**Proposed staging table 1:** `stg_bls__ep_entry_requirements`
- **Grain / key:** one row per `soc`, edition 2025. Key asserted unique; 831 rows.
- **Columns:** `edition` int (2025), `soc_code` text, `soc_title` text, `education` text, `education_rank` smallint, `experience` text, `experience_rank` smallint, `training` text, `training_rank` smallint, `has_ooh_page` bool, `source_file` text.

**Proposed staging table 2:** `stg_bls__ep_attainment`
- **Grain / key:** one row per `soc × attainment_level` (long, 7 levels), so it's easy to sum "bachelor's or more". 832 × 7 = 5,824 rows incl. the total.
- **Columns:** `edition` int, `attainment_years` text (`2023-24`), `soc_code` text, `attainment_level` text, `attainment_rank` smallint, `pct_of_workers` numeric, `acs_inexact` bool, `source_file` text.

Tables 5.1 and 5.2 are national summaries: not staged (5.2 can be rebuilt from the matrix if needed; 5.1 is a context sentence).

In [16]:
# Row accounting for the proposed staging tables, recomputed rather than typed
assert len(t54) == 831 and not t54.soc.duplicated().any()
assert len(t53) == 832 and not t53.soc.duplicated().any()
print(f"stg_bls__ep_entry_requirements: {len(t54)} rows | stg_bls__ep_attainment: {len(t53)} x {len(ATT)} = {len(t53) * len(ATT):,} rows")

stg_bls__ep_entry_requirements: 831 rows | stg_bls__ep_attainment: 832 x 7 = 5,824 rows
